# Privacy mode comparison

The same federated learning run under ppflx's twelve privacy modes, compared on
cost and model quality. Every number in this notebook is read from
`results/<dataset>/comparison_report.json`; [`results/README.md`](results/README.md)
describes the runs and the setup.

What each mode protects is stated in ppflx's
[SECURITY.md](https://github.com/CorpiXo/ppflx/blob/main/SECURITY.md); the proof
protocols are in [docs/ZKP.md](https://github.com/CorpiXo/ppflx/blob/main/docs/ZKP.md).
This notebook reports measurements; it does not establish security properties.

## 1. Setup

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ImportError:  # plain Python: print instead of rendering
    display = print
    Markdown = str

from ppflx_bench.compare.registry import MODES

RESULTS_DIR = Path(os.environ.get("PPFLX_RESULTS_DIR", "results"))
EXPORT_DIR = RESULTS_DIR / "comparison_analysis"

# Dataset order for tables and figures; any other dataset with a report follows.
DATASET_ORDER = ["healthcare", "creditcard", "stock", "mnist"]
DATASET_LABELS = {"healthcare": "Healthcare", "creditcard": "Credit card fraud",
                  "stock": "Stock", "mnist": "MNIST", "cifar10": "CIFAR-10"}
MODE_ORDER = list(MODES)
MODE_LABELS = {key: cfg.display_name for key, cfg in MODES.items()}

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## 2. What each mode protects

Taken from ppflx's SECURITY.md and the README's mode table. *Confidentiality* is
against the aggregation server; *integrity* means the server checks a proof about
each client's upload before aggregating it.

- **HE** — clients encrypt their update; the server aggregates ciphertexts and
  clients decrypt the average. The server still sees metadata such as each
  client's sample count (CKKS and ElGamal weight by it). CKKS (TenSEAL) is approximate; TFHE (Concrete)
  quantizes each weight to a 14-bit integer over [-5, 5] and adds them with a
  compiled circuit; the server holds only the circuit and evaluation keys.
  Clients share one secret key, so HE hides updates from the server, not from
  other clients. See [docs/FHE.md](docs/FHE.md).
- **ZKP** (`zkp`) — each client proves with Groth16 (gnark, BN254) that its
  update against the model it downloaded has a bounded L2 norm, over every
  coordinate. The server sees the plaintext update.
- **CKKS/TFHE + ZKP composites** — the proof covers a plaintext vector the
  client chooses and is **not bound** to the ciphertext the server aggregates: a
  client can prove an honest vector and upload another. They are
  confidentiality-only.
- **HE-ElGamal + ZKP** — exponential ElGamal on BabyJubJub. The proof is bound to
  the uploaded ciphertexts: each encrypts an in-range value, and the update
  against the encrypted global model is norm-bounded. The sampled variant proves
  server-sampled committed coordinates, so detection is probabilistic.
- **DP** (`dp` and the `_dp` composites) — during local training each batch's
  mean gradient is clipped to norm C and Gaussian noise with standard deviation
  σ·C/B is added (B is the batch size). σ is set from the recorded ε and δ for a
  single Gaussian-mechanism step; there is no per-example clipping and no
  accounting across steps or rounds, so the recorded ε is a noise setting, not an
  end-to-end (ε, δ) guarantee for the published model. See [docs/DP.md](docs/DP.md).

Limits that apply to every proof mode: the Groth16 setup is single-party (whoever
ran it could forge proofs), and a bounded update can still be malicious — the bound
caps a client's per-round influence, not its direction. In the ElGamal modes the
server also sees per-chunk update norms.

In [ ]:
# (confidentiality from the server, integrity checked by the server, DP noise)
MODE_PROPERTIES = {
    "baseline":                ("none",              "none",                                     False),
    "he_tenseal":              ("CKKS",              "none",                                     False),
    "he_concrete_tfhe":        ("TFHE",              "none",                                     False),
    "zkp":                     ("none",              "norm bound, every coordinate",             False),
    "zkp_sampled":             ("none",              "not claimed (sampled-cost benchmark)",     False),
    "dp":                      ("none",              "none",                                     True),
    "he_tenseal_zkp":          ("CKKS",              "none (proof not bound to ciphertext)",     False),
    "he_concrete_tfhe_zkp":    ("TFHE",              "none (proof not bound to ciphertext)",     False),
    "he_elgamal_zkp":          ("ElGamal",           "bound to ciphertext, every coordinate",    False),
    "he_elgamal_zkp_sampled":  ("ElGamal",           "bound to ciphertext, sampled coordinates", False),
    "he_tenseal_zkp_dp":       ("CKKS",              "none (proof not bound to ciphertext)",     True),
    "he_concrete_tfhe_zkp_dp": ("TFHE",              "none (proof not bound to ciphertext)",     True),
}


def protection_class(mode):
    conf, integ, dp = MODE_PROPERTIES[mode]
    parts = []
    if conf != "none":
        parts.append("confidentiality")
    if integ.startswith(("norm bound", "bound to")):
        parts.append("integrity")
    if dp:
        parts.append("DP noise")
    return " + ".join(parts) or "none"


properties = pd.DataFrame(
    [(MODE_LABELS.get(m, m), m, *MODE_PROPERTIES[m], protection_class(m)) for m in MODE_ORDER if m in MODE_PROPERTIES],
    columns=["Mode", "Key", "Confidentiality", "Integrity", "DP noise", "Class"],
).set_index("Mode")
display(properties)

## 3. How the results were produced

One full comparison per dataset, from `ppflx-bench/` (see the README's
"Benchmark on a new machine" for keys and the proof service). Without flags,
`compare.py` runs all twelve modes with 3 clients and 10 rounds over a networked
SuperLink, one SuperNode per client.

In [ ]:
for dataset in DATASET_ORDER:
    env = "FL_CONCRETE_TFHE_FORCE_REAL=1 " if dataset == "mnist" else ""
    print(f"{env}python compare.py --dataset {dataset}")

## 4. Load and validate the reports

A run counts only if every mode succeeded, every round aggregated (sampled modes:
committed, then aggregated) with no rejected clients and no Flower failures, and
ZKP validation passed. The table repeats those checks on the loaded reports.

In [ ]:
def load_reports(results_dir=RESULTS_DIR):
    found = sorted(p.parent.name for p in results_dir.glob("*/comparison_report.json"))
    order = [d for d in DATASET_ORDER if d in found] + [d for d in found if d not in DATASET_ORDER]
    return {d: json.loads((results_dir / d / "comparison_report.json").read_text()) for d in order}


def check_entry(entry):
    bm = entry.get("benchmark") or {}
    outcomes = bm.get("round_outcomes") or []
    problems = []
    if not entry.get("success"):
        problems.append("not successful")
    if bm.get("transport") != "network":
        problems.append(f"transport={bm.get('transport')}")
    for o in outcomes:
        if o.get("outcome") not in ("aggregated", "committed"):
            problems.append(f"round {o.get('round')}: {o.get('outcome')}")
        if o.get("rejected"):
            problems.append(f"round {o.get('round')}: rejected {sorted(o['rejected'])}")
        if o.get("flower_failures"):
            problems.append(f"round {o.get('round')}: {o['flower_failures']} Flower failures")
    zv = entry.get("zkp_validation")
    if zv is not None and not zv.get("ok"):
        problems.append("ZKP validation failed")
    return problems


reports = load_reports()
checks = []
for dataset, entries in reports.items():
    for e in entries:
        bm = e.get("benchmark") or {}
        problems = check_entry(e)
        checks.append({
            "Dataset": dataset,
            "Mode": e["mode"],
            "Clients": bm.get("num_clients"),
            "Rounds": bm.get("rounds"),
            "ZKP validated rounds": (e.get("zkp_validation") or {}).get("rounds_checked"),
            "Valid": not problems,
            "Problems": "; ".join(problems[:3]),
        })
checks = pd.DataFrame(checks)
display(checks.pivot(index="Mode", columns="Dataset", values="Valid").reindex(MODE_ORDER).dropna(how="all"))
invalid = checks[~checks["Valid"]]
if invalid.empty:
    print(f"All {len(checks)} mode runs across {len(reports)} datasets pass the acceptance checks.")
else:
    display(invalid)

## 5. One table of everything

Times are means per client per round (fit, encryption, decryption, proving) or per
round on the server (aggregation, proof verification). **Run time** is the measured
wall-clock duration of the whole run, and **overhead** is run time divided by the
baseline's run time for the same dataset. Accuracy, AUPRC, F1, precision and
recall come from each client evaluating the global model on its own validation
split (10% of its training shard) after every round, averaged over all clients
and rounds; the reports store them as `test_*`. The held-out test split is not
part of the reports.

In [ ]:
def _mean(d, key):
    return ((d or {}).get(key) or {}).get("mean") or 0.0


rows = []
for dataset, entries in reports.items():
    baseline = next((e for e in entries if e["mode"] == "baseline"), None)
    for e in entries:
        bm, mode = e["benchmark"], e["mode"]
        t, q = bm.get("timing", {}), bm.get("model_quality", {})
        proofs = ((e.get("zkp_validation") or {}).get("proofs_per_client") or [None])[0]
        prove = _mean(t, "proof_generation")
        rows.append({
            "Dataset": dataset,
            "Key": mode,
            "Mode": MODE_LABELS.get(mode, mode),
            "Class": protection_class(mode) if mode in MODE_PROPERTIES else "unknown",
            "Accuracy (%)": _mean(q, "test_accuracy"),
            "AUPRC (%)": _mean(q, "test_auprc"),
            "F1 (%)": _mean(q, "test_f1"),
            "Precision (%)": _mean(q, "test_precision"),
            "Recall (%)": _mean(q, "test_recall"),
            "Run time (s)": e["duration"],
            "Overhead (x)": e["duration"] / baseline["duration"] if baseline else np.nan,
            "Fit (s)": _mean(t, "client_fit"),
            "Encrypt (s)": _mean(t, "encryption"),
            "Decrypt (s)": _mean(t, "decryption"),
            "Proving (s)": prove,
            "Proofs per client-round": proofs,
            "Per proof (s)": prove / proofs if proofs else np.nan,
            "Verify (s)": _mean(t, "proof_verification"),
            "Aggregate (s)": _mean(t, "server_aggregate"),
            "Upload (KB)": _mean(bm.get("communication_bytes"), "upload") / 1024,
            "Download (KB)": _mean(bm.get("communication_bytes"), "download") / 1024,
            "Client peak (MB)": _mean(bm.get("memory_mb"), "client_peak"),
            "Server peak (MB)": _mean(bm.get("memory_mb"), "server_peak"),
        })
df = pd.DataFrame(rows)
df["Key"] = pd.Categorical(df["Key"], categories=MODE_ORDER, ordered=True)
df["_order"] = df["Dataset"].map({d: i for i, d in enumerate(reports)})
df = df.sort_values(["_order", "Key"]).drop(columns="_order").reset_index(drop=True)

# Primary quality metric: AUPRC for the imbalanced fraud data, accuracy otherwise.
PRIMARY = {d: ("AUPRC (%)" if d == "creditcard" else "Accuracy (%)") for d in reports}
base = df[df["Key"] == "baseline"].set_index("Dataset")
df["Quality"] = [r[PRIMARY[r["Dataset"]]] for _, r in df.iterrows()]
df["Δ quality vs baseline (pp)"] = [r["Quality"] - base.loc[r["Dataset"], PRIMARY[r["Dataset"]]] for _, r in df.iterrows()]
print(f"{len(df)} rows: {df['Key'].nunique()} modes × {df['Dataset'].nunique()} datasets")

### 5.1 Model quality

Modes that do not change the update — ZKP, CKKS, ElGamal and their sampled
variants — train the same model as the baseline and report the same metrics, up
to the fixed-point encoding of the update. Differences come from TFHE's 14-bit
quantization and from DP noise.

In [ ]:
QUALITY_COLS = ["Mode", "Accuracy (%)", "AUPRC (%)", "F1 (%)", "Precision (%)", "Recall (%)", "Δ quality vs baseline (pp)"]
for dataset in reports:
    d = df[df["Dataset"] == dataset]
    display(Markdown(f"**{DATASET_LABELS.get(dataset, dataset)}** — primary metric: {PRIMARY[dataset]}"))
    display(d[QUALITY_COLS].set_index("Mode").round(2))

### 5.2 Cost

In [ ]:
COST_COLS = ["Mode", "Run time (s)", "Overhead (x)", "Fit (s)", "Encrypt (s)", "Decrypt (s)", "Proving (s)",
             "Proofs per client-round", "Verify (s)", "Aggregate (s)", "Upload (KB)", "Client peak (MB)"]
for dataset in reports:
    d = df[df["Dataset"] == dataset]
    display(Markdown(f"**{DATASET_LABELS.get(dataset, dataset)}**"))
    display(d[COST_COLS].set_index("Mode").round(2))

## 6. Figures

In [ ]:
def per_dataset_bars(column, title, log=False, ylabel=None, value_fmt="{:.1f}"):
    n = len(reports)
    fig, axes = plt.subplots(n, 1, figsize=(11, 3.2 * n), squeeze=False)
    for ax, dataset in zip(axes[:, 0], reports):
        d = df[df["Dataset"] == dataset]
        col = PRIMARY[dataset] if column == "Quality" else column
        vals = d[col].astype(float)
        ax.bar(d["Mode"], vals, color=[CLASS_COLORS.get(c, "grey") for c in d["Class"]])
        if log:
            ax.set_yscale("log")
            ax.set_ylim(top=vals.max() * 4)
        else:
            ax.set_ylim(top=vals.max() * 1.2)
        for x, v in zip(d["Mode"], vals):
            if v > 0:
                ax.annotate(value_fmt.format(v), (x, v), ha="center", va="bottom", fontsize=7)
        ax.set_title(f"{DATASET_LABELS.get(dataset, dataset)}" + (f" — {col}" if column == "Quality" else ""), fontsize=10)
        ax.set_ylabel(ylabel or col)
        ax.tick_params(axis="x", rotation=35, labelsize=8)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CLASS_COLORS.values()]
    fig.legend(handles, CLASS_COLORS.keys(), loc="upper center", ncol=3, fontsize=8, bbox_to_anchor=(0.5, 1.02))
    fig.suptitle(title, y=1.05, fontweight="bold")
    fig.tight_layout()
    plt.show()


CLASS_COLORS = {
    "none": "#9e9e9e",
    "confidentiality": "#1f77b4",
    "integrity": "#2ca02c",
    "confidentiality + integrity": "#17becf",
    "DP noise": "#ff7f0e",
    "confidentiality + DP noise": "#9467bd",
}

### 6.1 Model quality by mode

Dots, not bars: each panel's axis is zoomed to its own range, so read the values,
not the heights. The dashed line is the baseline.

In [ ]:
n = len(reports)
fig, axes = plt.subplots(1, n, figsize=(4.4 * n, 5), squeeze=False, sharey=True)
for ax, dataset in zip(axes[0], reports):
    d = df[df["Dataset"] == dataset].iloc[::-1]
    metric = PRIMARY[dataset]
    ax.scatter(d[metric], d["Mode"], s=40, color=[CLASS_COLORS.get(c, "grey") for c in d["Class"]], zorder=3)
    ax.axvline(base.loc[dataset, metric], ls="--", color="black", lw=0.8)
    lo, hi = d[metric].min(), d[metric].max()
    pad = max(0.5, (hi - lo) * 0.15)
    ax.set_xlim(lo - pad, hi + pad)
    ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10)
    ax.set_xlabel(metric)
    ax.tick_params(axis="y", labelsize=8)
handles = [plt.Line2D([], [], marker="o", ls="", color=c) for c in CLASS_COLORS.values()]
fig.legend(handles, CLASS_COLORS.keys(), loc="upper center", ncol=6, fontsize=8, bbox_to_anchor=(0.5, 1.06))
fig.suptitle("Primary quality metric by mode", y=1.12, fontweight="bold")
fig.tight_layout(); plt.show()

### 6.2 Run time relative to the baseline (log scale)

In [ ]:
per_dataset_bars("Overhead (x)", "Measured run time ÷ baseline run time", log=True, value_fmt="{:.1f}×")

### 6.3 Upload per client per round (log scale)

In [ ]:
per_dataset_bars("Upload (KB)", "Bytes a client uploads per round", log=True, ylabel="KB", value_fmt="{:,.0f}")

### 6.4 Proving cost

Proving time grows with the number of proofs, which grows with the model: the
norm circuit covers 256 values per proof and the ElGamal circuit 128 coordinates.
The time per proof stays roughly constant across datasets, so the per-round cost
of a mode on a new model can be estimated from its parameter count. Times are per
client while all three clients prove at once against one prover on one machine,
so a single client proving alone would be faster.

In [ ]:
proof_modes = ["zkp", "he_elgamal_zkp"]
p = df[df["Key"].isin(proof_modes)].copy()
display(p.pivot_table(index="Dataset", columns="Mode", values=["Proofs per client-round", "Proving (s)", "Per proof (s)"], observed=True)
         .reindex(list(reports)).round(2))

fig, ax = plt.subplots(figsize=(7, 4.5))
from matplotlib.ticker import ScalarFormatter
for mode, marker in zip(proof_modes, ["o", "s"]):
    d = p[p["Key"] == mode]
    ax.scatter(d["Proofs per client-round"], d["Proving (s)"], marker=marker, s=60, label=MODE_LABELS[mode])
    # Datasets with the same model size share a point: label it once.
    for proofs, g in d.groupby("Proofs per client-round"):
        ax.annotate(", ".join(g["Dataset"]), (proofs, g["Proving (s)"].mean()), textcoords="offset points", xytext=(6, -10), fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log")
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(ScalarFormatter())
    axis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("Proofs per client per round"); ax.set_ylabel("Proving time per client per round (s)")
ax.set_title("Proving time against number of proofs", fontweight="bold")
ax.legend(); fig.tight_layout(); plt.show()

### 6.5 Cost against quality

Each point is a mode; the x axis is measured overhead, the y axis the change in
the primary quality metric against the baseline. The colour is what the mode
protects (section 2), not a score: the modes protect different things and are not
ranked on a single scale.

In [ ]:
n = len(reports)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4.2), squeeze=False)
for ax, dataset in zip(axes[0], reports):
    d = df[df["Dataset"] == dataset]
    for cls, g in d.groupby("Class"):
        ax.scatter(g["Overhead (x)"], g["Δ quality vs baseline (pp)"], s=45, color=CLASS_COLORS.get(cls, "grey"), label=cls)
    ax.axhline(0, color="black", lw=0.7)
    ax.set_xscale("log")
    ax.set_title(DATASET_LABELS.get(dataset, dataset), fontsize=10)
    ax.set_xlabel("Run time ÷ baseline")
    ax.set_ylabel(f"Δ {PRIMARY[dataset]} (pp)")
present = set(df["Class"])
handles = [plt.Line2D([], [], marker="o", ls="", color=c) for k, c in CLASS_COLORS.items() if k in present]
fig.legend(handles, [k for k in CLASS_COLORS if k in present], loc="upper center", ncol=6, fontsize=8, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); plt.show()

## 7. Findings

Computed from the reports above, so they follow the data when the results change.

In [ ]:
def fmt_kb(kb):
    return f"{kb / 1024:,.1f} MB" if kb >= 1024 else f"{kb:,.1f} KB"


lines = []
for dataset in reports:
    d = df[df["Dataset"] == dataset].set_index("Key", drop=False)
    label = DATASET_LABELS.get(dataset, dataset)
    metric = PRIMARY[dataset]
    same = ["zkp", "zkp_sampled", "he_tenseal", "he_tenseal_zkp", "he_elgamal_zkp", "he_elgamal_zkp_sampled"]
    same = [m for m in same if m in d.index]
    spread = d.loc[same, metric].max() - d.loc[same, metric].min()
    lines.append(f"**{label}** ({metric}, baseline {d.loc['baseline', metric]:.2f}%)")
    lines.append(f"- Modes that keep the update unchanged stay within {spread:.2f} pp of each other.")
    for m in ("he_concrete_tfhe", "dp"):
        if m in d.index:
            lines.append(f"- {MODE_LABELS[m]}: {d.loc[m, 'Δ quality vs baseline (pp)']:+.2f} pp.")
    slow = d.loc[d["Overhead (x)"].idxmax()]
    lines.append(f"- Slowest: {slow['Mode']}, {slow['Overhead (x)']:.1f}× the baseline run time "
                 f"({slow['Run time (s)'] / 3600:.1f} h).")
    for m in ("he_elgamal_zkp", "zkp"):
        if m in d.index:
            lines.append(f"- {MODE_LABELS[m]}: {d.loc[m, 'Overhead (x)']:.1f}× run time, "
                         f"{d.loc[m, 'Proving (s)']:.0f} s proving per client per round.")
    up = d["Upload (KB)"]
    lines.append(f"- Upload per client per round: baseline {fmt_kb(up['baseline'])}, "
                 + ", ".join(f"{MODE_LABELS[m]} {fmt_kb(up[m])} ({up[m] / up['baseline']:,.0f}×)"
                             for m in ("he_tenseal", "he_concrete_tfhe", "he_elgamal_zkp") if m in up.index) + ".")
    lines.append("")
display(Markdown("\n".join(lines)))

**Reading the numbers**

- Credit card fraud is about 0.17% positive, so accuracy is near 100% for any
  model; AUPRC and F1 are the metrics to compare.
- Stock is close to chance for every mode and does not separate modes by quality.
- `he_elgamal_zkp` encrypts inside the proof service, so its encryption time is
  part of proving; the sampled variant encrypts at commit time.
- Timings come from one laptop CPU (see `results/README.md`) and one run per
  dataset; treat small differences in run time as noise.

## 8. Export

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(EXPORT_DIR / "all_modes.csv", index=False)
properties.to_csv(EXPORT_DIR / "mode_properties.csv")
print(f"Wrote {EXPORT_DIR / 'all_modes.csv'} and {EXPORT_DIR / 'mode_properties.csv'}")